In [59]:
import pandas as pd 
df = pd.read_excel("2025_TIF_Raw_Data.xlsx") 
df

,County,CoMun,TVC,Municipality,TID,Base Year,Current Value,Base Value,Increment
0,ADAMS,1030,TOWN,ROME,001T,2015,163622200,1249400,162372800
1,ADAMS,1291,CITY,WISCONSIN DELLS,003,2005,72515000,2038600,70476400
2,ASHLAND,2201,CITY,ASHLAND,010,2017,13564100,3937900,9626200
3,BARRON,3111,VILLAGE,CAMERON,001,2005,24328400,2317500,22010900
4,BARRON,3116,VILLAGE,DALLAS,002,2001,2286400,29900,2256500
...,...,...,...,...,...,...,...,...,...
1440,WOOD,71261,CITY,NEKOOSA,003,2012,31265700,16204500,15061200
1441,WOOD,71261,CITY,NEKOOSA,004,2018,8023100,3086000,4937100
1442,WOOD,71291,CITY,WISCONSIN RAPIDS,006,2004,19236000,3812800,15423200
1443,WOOD,71291,CITY,WISCONSIN RAPIDS,007,2005,58649500,31842200,26807300


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1445 entries, 0 to 1444
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   County          1445 non-null   str  
 1   CoMun           1445 non-null   int64
 2   TVC             1445 non-null   str  
 3   Municipality    1445 non-null   str  
 4   TID             1445 non-null   str  
 5    Base Year      1445 non-null   int64
 6    Current Value  1445 non-null   int64
 7    Base Value     1445 non-null   int64
 8    Increment      1445 non-null   int64
dtypes: int64(5), str(4)
memory usage: 101.7 KB


In [27]:
# ── ONE-TIME FIXES ────────────────────────────────────────────────────────────
# Strip whitespace from string columns
str_cols = ["County", "Municipality", "TVC", "TID"]
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

# Base Year just needs the leading space removed + cast to int
df["Base Year"] = df["Base Year"].astype(int)

# Rename to drop all leading spaces
df.rename(columns={
    " Current Value": "Current Value",
    " Base Value":    "Base Value",
    " Increment":     "Increment",
    " Base Year":     "Base Year"
}, inplace=True)

# Verify dtypes look correct before proceeding
print(df.dtypes)
print(df.head(2))


# ── STEP 1: Isolate Brookfield TID1 ──────────────────────────────────────────
brookfield_tid1 = df[
    (df["County"] == "WAUKESHA") &
    (df["Municipality"].str.contains("BROOKFIELD")) &
    (df["TVC"] == "TOWN") &
    (df["TID"].str.contains("001A"))
]
brookfield_tid1

County             str
CoMun            int64
TVC                str
Municipality       str
TID                str
Base Year        int64
Current Value    int64
Base Value       int64
Increment        int64
dtype: object
  County  CoMun   TVC     Municipality   TID  Base Year  Current Value  \
0  ADAMS   1030  TOWN             ROME  001T       2015      163622200   
1  ADAMS   1291  CITY  WISCONSIN DELLS   003       2005       72515000   

   Base Value  Increment  
0     1249400  162372800  
1     2038600   70476400  


,County,CoMun,TVC,Municipality,TID,Base Year,Current Value,Base Value,Increment
1285,WAUKESHA,67002,TOWN,BROOKFIELD,001A,2014,363309400,62972100,300337300


In [41]:
# ── STEP 2: Statewide Ranking ─────────────────────────────────────────────────
# Total increment across all Wisconsin TIDs
total_increment = df["Increment"].sum()
print(f"Total statewide increment: ${total_increment:,.0f}")

## ___________
# Percentile rank — 0.97 means larger than 97% of all TIDs
df["Percentile"] = df["Increment"].rank(pct=True)
tid1_percentile = df.loc[brookfield_tid1.index, "Percentile"].iloc[0]
print(f"Brookfield TID1 percentile: {tid1_percentile:.1%}")


Total statewide increment: $46,675,718,763
Brookfield TID1 percentile: 99.2%


In [49]:
## ____________
# ── STEP 3: Waukesha County Concentration ────────────────────────────────────
# All Waukesha TIDs sorted by increment
waukesha = df[df["County"] == "WAUKESHA"]
waukesha.sort_values("Increment", ascending=False)

,County,CoMun,TVC,Municipality,TID,Base Year,Current Value,Base Value,Increment,Percentile
1285,WAUKESHA,67002,TOWN,BROOKFIELD,001A,2014,363309400,62972100,300337300,0.991696
1299,WAUKESHA,67151,VILLAGE,MENOMONEE FALLS,008,2008,193829400,19443200,174386200,0.968858
1294,WAUKESHA,67147,VILLAGE,LANNON,001,2018,151147200,10681500,140465700,0.952941
1334,WAUKESHA,67291,CITY,WAUKESHA,011,1997,150387800,33833500,116554300,0.939792
1324,WAUKESHA,67251,CITY,MUSKEGO,010,2008,115105900,1150600,113955300,0.937024
...,...,...,...,...,...,...,...,...,...,...
1288,WAUKESHA,67116,VILLAGE,DOUSMAN,002,2024,25771000,24730900,1040100,0.132180
1312,WAUKESHA,67153,VILLAGE,MUKWONAGO,006,2023,846300,846300,0,0.037716
1333,WAUKESHA,67265,CITY,OCONOMOWOC,008,2023,472900,493900,-21000,0.024913
1309,WAUKESHA,67151,VILLAGE,MENOMONEE FALLS,018,2024,0,732300,-732300,0.008304


In [51]:
# TID1's share of the entire county's increment
tid1_increment = brookfield_tid1["Increment"].iloc[0]
waukesha_total = waukesha["Increment"].sum()
tid1_share     = tid1_increment / waukesha_total
print(f"TID1 share of Waukesha County total: {tid1_share:.1%}")

TID1 share of Waukesha County total: 10.5%


In [53]:
# ── STEP 4: Statewide Outliers ────────────────────────────────────────────────
# All TIDs over $250M — Brookfield TID1's true peers
large_tids = df[df["Increment"] > 250_000_000]
print(large_tids[["Municipality", "TVC", "Base Year", "Increment"]]
      .sort_values("Increment", ascending=False))

          Municipality      TVC  Base Year   Increment
1011    MOUNT PLEASANT  VILLAGE       2018  1701377600
235            MADISON     CITY       2005   686485600
249          MIDDLETON     CITY       1993   676238400
27         ASHWAUBENON  VILLAGE       2008   536957000
241            MADISON     CITY       2015   518909300
734          MILWAUKEE     CITY       2002   435967600
1123       LAKE DELTON  VILLAGE       2005   432888200
756          MILWAUKEE     CITY       2013   364875600
500   PLEASANT PRAIRIE  VILLAGE       2017   359450600
242            MADISON     CITY       2021   338381400
753          MILWAUKEE     CITY       2009   316725500
32              HOBART  VILLAGE       2009   312846700
1285        BROOKFIELD     TOWN       2014   300337300
244            MADISON     CITY       2022   288941600
654             WESTON  VILLAGE       1998   282535100
250          MIDDLETON     CITY       2009   282095700
494            BRISTOL  VILLAGE       2019   274881100
739       

In [55]:
# ── STEP 5: Age vs. Size Correlation ─────────────────────────────────────────
# Does age explain size? Correlation near 1.0 = yes, near 0 = something else is driving it
df["Age"] = 2025 - df["Base Year"]
print(df[["Age", "Increment"]].corr())

# Brookfield TID1's specific age
tid1_age = df.loc[brookfield_tid1.index, "Age"].iloc[0]
print(f"Brookfield TID1 age: {tid1_age} years")

                Age  Increment
Age        1.000000   0.119076
Increment  0.119076   1.000000
Brookfield TID1 age: 11 years


In [57]:
# Check if .str.contains("BROOKFIELD") catches anything unintended
print(df[df["Municipality"].str.contains("BROOKFIELD")]["Municipality"].unique())

# Check if "001A" is truly unique — could another municipality have a TID 001A?
print(df[df["TID"].str.contains("001A")][["Municipality", "TVC", "County", "TID"]])

<StringArray>
['BROOKFIELD']
Length: 1, dtype: str
     Municipality   TVC     County   TID
20       LAWRENCE  TOWN      BROWN  001A
23      LEDGEVIEW  TOWN      BROWN  001A
311     GIBRALTAR  TOWN       DOOR  001A
449        IXONIA  TOWN  JEFFERSON  001A
855      MINOCQUA  TOWN     ONEIDA  001A
863       FREEDOM  TOWN  OUTAGAMIE  001A
866   GRAND CHUTE  TOWN  OUTAGAMIE  001A
1167    SHEBOYGAN  TOWN  SHEBOYGAN  001A
1285   BROOKFIELD  TOWN   WAUKESHA  001A
1286   OCONOMOWOC  TOWN   WAUKESHA  001A
1369       ALGOMA  TOWN  WINNEBAGO  001A
1370      CLAYTON  TOWN  WINNEBAGO  001A


In [1]:
import pandas as pd

# ─────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────
df = pd.read_excel("2025_TIF_Raw_Data.xlsx")

# Normalize column names (removes hidden leading/trailing spaces)
df.columns = df.columns.str.strip()

# ─────────────────────────────────────────────────────────────
# CLEAN DATA TYPES
# ─────────────────────────────────────────────────────────────

# Clean string columns safely
for col in ["County", "Municipality", "TVC", "TID"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Ensure numeric columns are properly typed
df["Base Year"] = pd.to_numeric(df["Base Year"], errors="coerce").astype("Int64")
df["Increment"] = pd.to_numeric(df["Increment"], errors="coerce")
df["Base Value"] = pd.to_numeric(df["Base Value"], errors="coerce")
df["Current Value"] = pd.to_numeric(df["Current Value"], errors="coerce")

# Drop rows where Increment is missing (protect calculations)
df = df.dropna(subset=["Increment"])

print("Data types after cleaning:")
print(df.dtypes)
print(df.head(2))


# ─────────────────────────────────────────────────────────────
# STEP 1: Isolate Brookfield TID1
# ─────────────────────────────────────────────────────────────

brookfield_tid1 = df[
    (df["County"] == "WAUKESHA") &
    (df["Municipality"].str.contains("BROOKFIELD", case=False, regex=False)) &
    (df["TVC"] == "TOWN") &
    (df["TID"].str.contains("001A", case=False, regex=False))
]

assert len(brookfield_tid1) == 1, "Unexpected duplicate or missing Brookfield TID1"

print("\nBrookfield TID1 record:")
print(brookfield_tid1)


# ─────────────────────────────────────────────────────────────
# STEP 2: Statewide Ranking
# ─────────────────────────────────────────────────────────────

total_increment = df["Increment"].sum()
print(f"\nTotal statewide increment: ${total_increment:,.0f}")

# Percentile ranking (ascending — highest increment = near 1.0)
df["Percentile"] = df["Increment"].rank(pct=True, ascending=True)

tid1_percentile = df.loc[brookfield_tid1.index, "Percentile"].values[0]
print(f"Brookfield TID1 percentile statewide: {tid1_percentile:.1%}")


# ─────────────────────────────────────────────────────────────
# STEP 3: Waukesha County Concentration
# ─────────────────────────────────────────────────────────────

waukesha = df[df["County"] == "WAUKESHA"]

waukesha_sorted = waukesha.sort_values("Increment", ascending=False)

print("\nTop Waukesha County TIDs by Increment:")
print(waukesha_sorted.head(10)[["Municipality", "TVC", "TID", "Increment"]])

waukesha_total = waukesha["Increment"].sum()
tid1_increment = brookfield_tid1["Increment"].values[0]

tid1_share = tid1_increment / waukesha_total

print(f"\nTID1 share of Waukesha County total increment: {tid1_share:.1%}")


# ─────────────────────────────────────────────────────────────
# STEP 4: Statewide Outliers
# ─────────────────────────────────────────────────────────────

large_tids = df[df["Increment"] > 250_000_000]

print("\nStatewide TIDs over $250M increment:")
print(
    large_tids[["County", "Municipality", "TVC", "Base Year", "Increment"]]
    .sort_values("Increment", ascending=False)
)


# ─────────────────────────────────────────────────────────────
# STEP 5: Age vs. Size Correlation
# ─────────────────────────────────────────────────────────────

current_year = 2025
df["Age"] = current_year - df["Base Year"]

correlation = df["Age"].corr(df["Increment"])
print(f"\nCorrelation between TID age and increment: {correlation:.3f}")

tid1_age = df.loc[brookfield_tid1.index, "Age"].values[0]
print(f"Brookfield TID1 age: {tid1_age} years")


# ─────────────────────────────────────────────────────────────
# VALIDATION CHECKS
# ─────────────────────────────────────────────────────────────

print("\nMunicipalities containing 'BROOKFIELD':")
print(df[df["Municipality"].str.contains("BROOKFIELD", case=False, regex=False)]
      ["Municipality"].unique())

print("\nAll TIDs containing '001A':")
print(df[df["TID"].str.contains("001A", case=False, regex=False)]
      [["County", "Municipality", "TVC", "TID"]])

Data types after cleaning:
County             str
CoMun            int64
TVC                str
Municipality       str
TID                str
Base Year        Int64
Current Value    int64
Base Value       int64
Increment        int64
dtype: object
  County  CoMun   TVC     Municipality   TID  Base Year  Current Value  \
0  ADAMS   1030  TOWN             ROME  001T       2015      163622200   
1  ADAMS   1291  CITY  WISCONSIN DELLS   003       2005       72515000   

   Base Value  Increment  
0     1249400  162372800  
1     2038600   70476400  

Brookfield TID1 record:
        County  CoMun   TVC Municipality   TID  Base Year  Current Value  \
1285  WAUKESHA  67002  TOWN   BROOKFIELD  001A       2014      363309400   

      Base Value  Increment  
1285    62972100  300337300  

Total statewide increment: $46,675,718,763
Brookfield TID1 percentile statewide: 99.2%

Top Waukesha County TIDs by Increment:
         Municipality      TVC   TID  Increment
1285       BROOKFIELD     TOWN  00

In [13]:
import pandas as pd

## First let's import our files 

files = {
    2019: "2019_TIF_Raw_Data.xlsx",
    2020: "2020_TIF_Raw_Data.xlsx",
    2021: "2021_TIF_Raw_Data.xlsx",
    2022: "2022_TIF_Raw_Data.xlsx",
    2023: "2023_TIF_Raw_Data.xlsx",
    2024: "2024_TIF_Raw_Data.xlsx",
    2025: "2025_TIF_Raw_Data.xlsx"
}

# Store Brookfield results here
results = []

## Column mapping to ensure consistency
column_map = {
    "County": "COUNTY",
    "CoMun": "COMUN",
    "TVC": "TVC",
    "Municipality": "MUNICIPALITY",
    "TID #": "TID_NUMBER",
    "Base Year": "BASE_YEAR",
    "Current Value": "CURRENT_VALUE",
    "Base Value": "BASE_VALUE",
    "Increment": "INCREMENT"
}

## Now let's loop through each year 
for year, file in files.items():

    print(f"\nProcessing {year}...")

    df = pd.read_excel(file)

    ## Normalize column names: strip spaces, uppercase, replace '#' and ' '
    df.columns = (
        df.columns
        .str.strip()
        .str.replace("#", "NUMBER", regex=False)
        .str.replace(" ", "_", regex=False)
        .str.upper()
        .str.normalize('NFKD')  # remove special unicode characters
    )

    ## Apply explicit column map (handles hidden/inconsistent headers)
    df = df.rename(columns={k.upper(): v for k, v in column_map.items() if k.upper() in df.columns})

    ## Clean key columns
    for col in ["COUNTY", "MUNICIPALITY", "TVC", "TID_NUMBER", "COMUN"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    ## Convert numeric columns
    for col in ["BASE_YEAR", "INCREMENT", "BASE_VALUE", "CURRENT_VALUE"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    ## Drop rows missing Increment
    df = df.dropna(subset=["INCREMENT"])

    ## Statewide rankings
    df["STATEWIDE_PERCENTILE"] = df["INCREMENT"].rank(pct=True, ascending=True)
    df["STATEWIDE_RANK"] = df["INCREMENT"].rank(ascending=False, method="min").astype(int)

    ## County rankings
    df["COUNTY_PERCENTILE"] = df.groupby("COUNTY")["INCREMENT"].rank(pct=True, ascending=True)
    df["COUNTY_RANK"] = df.groupby("COUNTY")["INCREMENT"].rank(ascending=False, method="min").astype(int)

    ## Calculate age
    df["AGE"] = year - df["BASE_YEAR"]

    ## State and county totals
    statewide_total = df["INCREMENT"].sum()
    county_totals = df.groupby("COUNTY")["INCREMENT"].sum()

    ## Isolate Brookfield TID1 using municipal code 67002
    tid1 = df[
        (df["COMUN"].astype(str) == "67002") &
        (df["TID_NUMBER"].astype(str).str.contains("001A", case=False, regex=False))
    ]

    ## Debug check (uncomment if needed)
    # print(df[df["COMUN"].astype(str) == "67002"][["MUNICIPALITY","TID_NUMBER"]])

    if len(tid1) == 1:
        row = tid1.iloc[0]
        county_total = county_totals[row["COUNTY"]]

        share_state = row["INCREMENT"] / statewide_total
        share_county = row["INCREMENT"] / county_total

        results.append({
            "Year": year,
            "Increment": row["INCREMENT"],

            "Statewide_Rank": row["STATEWIDE_RANK"],
            "Statewide_Percentile": row["STATEWIDE_PERCENTILE"],

            "County_Rank": row["COUNTY_RANK"],
            "County_Percentile": row["COUNTY_PERCENTILE"],

            "Age": row["AGE"],

            "Share_of_State": share_state,
            "Share_of_County": share_county
        })
    else:
        print(f"WARNING: Brookfield TID1 not uniquely found for {year}")

## Create final dataframe 
brookfield_timeline = pd.DataFrame(results)
brookfield_timeline = brookfield_timeline.sort_values("Year")

print("\nBrookfield TID1 Timeline:")
print(brookfield_timeline)

### Export Dataset 
brookfield_timeline.to_excel(
    "Brookfield_TID1_Timeline.xlsx",
    index=False
)

print("\nTimeline exported to Brookfield_TID1_Timeline.xlsx")


Processing 2019...

Processing 2020...

Processing 2021...

Processing 2022...


KeyError: 'BASE_YEAR'

In [15]:
import pandas as pd

## Files to process
files = {
    2019: "2019_TIF_Raw_Data.xlsx",
    2020: "2020_TIF_Raw_Data.xlsx",
    2021: "2021_TIF_Raw_Data.xlsx",
    2022: "2022_TIF_Raw_Data.xlsx",
    2023: "2023_TIF_Raw_Data.xlsx",
    2024: "2024_TIF_Raw_Data.xlsx",
    2025: "2025_TIF_Raw_Data.xlsx"
}

results = []

## Loop through each year
for year, file in files.items():
    print(f"\nProcessing {year}...")

    df = pd.read_excel(file)

    # Show actual columns in the file
    print("Raw columns:", df.columns.tolist())

    # Normalize columns: remove leading/trailing spaces, replace #, replace spaces, uppercase
    df.columns = df.columns.str.strip().str.replace("#", "NUMBER", regex=False).str.replace(" ", "_", regex=False).str.upper()

    print("Normalized columns:", df.columns.tolist())

    # Safe column mapping (map only if the column exists)
    column_map = {}
    for col in df.columns:
        if "BASE_YEAR" in col or "BASE YEAR" in col:
            column_map[col] = "BASE_YEAR"
        elif "INCREMENT" in col:
            column_map[col] = "INCREMENT"
        elif "BASE_VALUE" in col:
            column_map[col] = "BASE_VALUE"
        elif "CURRENT_VALUE" in col:
            column_map[col] = "CURRENT_VALUE"
        elif "TID_NUMBER" in col or "TID" in col:
            column_map[col] = "TID_NUMBER"
        elif "COMUN" in col or "CoMun".upper() in col:
            column_map[col] = "COMUN"
        elif "COUNTY" in col:
            column_map[col] = "COUNTY"
        elif "MUNICIPALITY" in col:
            column_map[col] = "MUNICIPALITY"
        elif "TVC" in col:
            column_map[col] = "TVC"

    df = df.rename(columns=column_map)

    # Convert numeric columns
    for col in ["BASE_YEAR", "INCREMENT", "BASE_VALUE", "CURRENT_VALUE"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop rows missing Increment
    df = df.dropna(subset=["INCREMENT"])

    # Statewide rankings
    df["STATEWIDE_PERCENTILE"] = df["INCREMENT"].rank(pct=True, ascending=True)
    df["STATEWIDE_RANK"] = df["INCREMENT"].rank(ascending=False, method="min").astype(int)

    # County rankings
    df["COUNTY_PERCENTILE"] = df.groupby("COUNTY")["INCREMENT"].rank(pct=True, ascending=True)
    df["COUNTY_RANK"] = df.groupby("COUNTY")["INCREMENT"].rank(ascending=False, method="min").astype(int)

    # Calculate age
    if "BASE_YEAR" in df.columns:
        df["AGE"] = year - df["BASE_YEAR"]
    else:
        df["AGE"] = None

    # State and county totals
    statewide_total = df["INCREMENT"].sum()
    county_totals = df.groupby("COUNTY")["INCREMENT"].sum()

    # Isolate Brookfield TID1 using municipal code 67002
    tid1 = df[
        (df.get("COMUN", pd.Series()) == "67002") &
        (df.get("TID_NUMBER", pd.Series()).astype(str).str.contains("001A", case=False, regex=False))
    ]

    if len(tid1) == 1:
        row = tid1.iloc[0]
        county_total = county_totals[row["COUNTY"]]

        share_state = row["INCREMENT"] / statewide_total
        share_county = row["INCREMENT"] / county_total

        results.append({
            "Year": year,
            "Increment": row["INCREMENT"],
            "Statewide_Rank": row["STATEWIDE_RANK"],
            "Statewide_Percentile": row["STATEWIDE_PERCENTILE"],
            "County_Rank": row["COUNTY_RANK"],
            "County_Percentile": row["COUNTY_PERCENTILE"],
            "Age": row["AGE"],
            "Share_of_State": share_state,
            "Share_of_County": share_county
        })
    else:
        print(f"WARNING: Brookfield TID1 not uniquely found for {year}")

# Final dataframe
brookfield_timeline = pd.DataFrame(results)
brookfield_timeline = brookfield_timeline.sort_values("Year")

print("\nBrookfield TID1 Timeline:")
print(brookfield_timeline)

# Export
brookfield_timeline.to_excel("Brookfield_TID1_Timeline.xlsx", index=False)
print("\nTimeline exported to Brookfield_TID1_Timeline.xlsx")


Processing 2019...
Raw columns: ['County', ' CoMun', ' TVC', ' Municipality', ' TID #', 'Base Year', ' Current Value', ' Base Value', ' Increment']
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2020...
Raw columns: ['County', ' CoMun', ' TVC', ' Municipality', ' TID #', ' Base Year', ' Current Value', ' Base Value', ' Increment']
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2021...
Raw columns: ['County', ' CoMun', ' TVC', ' Municipality', ' TID #', ' Base Year', ' Current Value', ' Base Value', ' Increment']
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2022...
Raw columns: ['County', ' CoMun', ' TVC', ' Municipality', ' TID #', ' Base Yr.', ' Current Value', ' Base Value', ' Increment']


KeyError: 'Year'

In [17]:
import pandas as pd

## Files to process
files = {
    2019: "2019_TIF_Raw_Data.xlsx",
    2020: "2020_TIF_Raw_Data.xlsx",
    2021: "2021_TIF_Raw_Data.xlsx",
    2022: "2022_TIF_Raw_Data.xlsx",
    2023: "2023_TIF_Raw_Data.xlsx",
    2024: "2024_TIF_Raw_Data.xlsx",
    2025: "2025_TIF_Raw_Data.xlsx"
}

results = []

for year, file in files.items():
    print(f"\nProcessing {year}...")

    df = pd.read_excel(file)

    # Normalize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.replace("#", "NUMBER", regex=False)
        .str.replace(" ", "_", regex=False)
        .str.upper()
    )
    print("Normalized columns:", df.columns.tolist())

    # Automatically map any variant of Base Year to BASE_YEAR
    base_year_variants = [col for col in df.columns if "BASE" in col and "YEAR" in col]
    if base_year_variants:
        df = df.rename(columns={base_year_variants[0]: "BASE_YEAR"})

    # Ensure consistent column names
    column_map = {
        "COUNTY": "COUNTY",
        "COMUN": "COMUN",
        "TVC": "TVC",
        "MUNICIPALITY": "MUNICIPALITY",
        "TID_NUMBER": "TID_NUMBER",
        "INCREMENT": "INCREMENT",
        "BASE_VALUE": "BASE_VALUE",
        "CURRENT_VALUE": "CURRENT_VALUE"
    }
    df = df.rename(columns={k: v for k, v in column_map.items() if k in df.columns})

    # Convert numeric columns safely
    for col in ["BASE_YEAR", "INCREMENT", "BASE_VALUE", "CURRENT_VALUE"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["INCREMENT"])

    # Rankings
    df["STATEWIDE_PERCENTILE"] = df["INCREMENT"].rank(pct=True, ascending=True)
    df["STATEWIDE_RANK"] = df["INCREMENT"].rank(ascending=False, method="min").astype(int)
    df["COUNTY_PERCENTILE"] = df.groupby("COUNTY")["INCREMENT"].rank(pct=True, ascending=True)
    df["COUNTY_RANK"] = df.groupby("COUNTY")["INCREMENT"].rank(ascending=False, method="min").astype(int)

    # Age
    if "BASE_YEAR" in df.columns:
        df["AGE"] = year - df["BASE_YEAR"]
    else:
        df["AGE"] = None

    # Totals
    statewide_total = df["INCREMENT"].sum()
    county_totals = df.groupby("COUNTY")["INCREMENT"].sum()

    # Isolate Brookfield TID1 using municipal code 67002
    tid1 = df[(df.get("COMUN", pd.Series()) == "67002") &
              (df.get("TID_NUMBER", pd.Series()).astype(str).str.contains("001A", case=False, regex=False))]

    if len(tid1) == 1:
        row = tid1.iloc[0]
        county_total = county_totals[row["COUNTY"]]

        results.append({
            "Year": year,
            "Increment": row["INCREMENT"],
            "Statewide_Rank": row["STATEWIDE_RANK"],
            "Statewide_Percentile": row["STATEWIDE_PERCENTILE"],
            "County_Rank": row["COUNTY_RANK"],
            "County_Percentile": row["COUNTY_PERCENTILE"],
            "Age": row["AGE"],
            "Share_of_State": row["INCREMENT"] / statewide_total,
            "Share_of_County": row["INCREMENT"] / county_total
        })
    else:
        print(f"WARNING: Brookfield TID1 not uniquely found for {year}, found {len(tid1)} rows")

# Final dataframe
if results:
    brookfield_timeline = pd.DataFrame(results)
    brookfield_timeline = brookfield_timeline.sort_values("Year")
    print("\nBrookfield TID1 Timeline:")
    print(brookfield_timeline)

    # Export
    brookfield_timeline.to_excel("Brookfield_TID1_Timeline.xlsx", index=False)
    print("\nTimeline exported to Brookfield_TID1_Timeline.xlsx")
else:
    print("\nNo valid Brookfield TID1 data found in any year.")


Processing 2019...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2020...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2021...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2022...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YR.', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2023...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2024...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2025...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPAL

This code processes annual TIF (Tax Increment Financing) Excel data from 2019–2025, standardizes and cleans the columns, calculates rankings, percentiles, age, and share of totals, and then extracts a year-by-year profile specifically for Brookfield TID1, exporting the results to an Excel timeline.

In [19]:
import pandas as pd

## Files to process
files = {
    2019: "2019_TIF_Raw_Data.xlsx",
    2020: "2020_TIF_Raw_Data.xlsx",
    2021: "2021_TIF_Raw_Data.xlsx",
    2022: "2022_TIF_Raw_Data.xlsx",
    2023: "2023_TIF_Raw_Data.xlsx",
    2024: "2024_TIF_Raw_Data.xlsx",
    2025: "2025_TIF_Raw_Data.xlsx"
}

results = []

for year, file in files.items():
    print(f"\nProcessing {year}...")

    df = pd.read_excel(file)

    # Normalize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.replace("#", "NUMBER", regex=False)
        .str.replace(" ", "_", regex=False)
        .str.upper()
    )
    print("Normalized columns:", df.columns.tolist())

    # Map any Base Year variant to BASE_YEAR
    base_year_candidates = [col for col in df.columns if "BASE" in col and "YR" in col or "YEAR" in col]
    if base_year_candidates:
        df = df.rename(columns={base_year_candidates[0]: "BASE_YEAR"})

    # Convert key columns to string and strip spaces
    for col in ["COMUN", "TID_NUMBER", "COUNTY", "MUNICIPALITY", "TVC"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    # Convert numeric columns
    for col in ["BASE_YEAR", "INCREMENT", "BASE_VALUE", "CURRENT_VALUE"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["INCREMENT"])

    # Rankings
    df["STATEWIDE_PERCENTILE"] = df["INCREMENT"].rank(pct=True, ascending=True)
    df["STATEWIDE_RANK"] = df["INCREMENT"].rank(ascending=False, method="min").astype(int)
    df["COUNTY_PERCENTILE"] = df.groupby("COUNTY")["INCREMENT"].rank(pct=True, ascending=True)
    df["COUNTY_RANK"] = df.groupby("COUNTY")["INCREMENT"].rank(ascending=False, method="min").astype(int)

    # Age
    if "BASE_YEAR" in df.columns:
        df["AGE"] = year - df["BASE_YEAR"]
    else:
        df["AGE"] = None

    # Totals
    statewide_total = df["INCREMENT"].sum()
    county_totals = df.groupby("COUNTY")["INCREMENT"].sum()

    # Isolate Brookfield TID1 using municipal code 67002 and TID # 001A
    tid1 = df[(df.get("COMUN") == "67002") &
              (df.get("TID_NUMBER") == "001A")]

    if len(tid1) == 1:
        row = tid1.iloc[0]
        county_total = county_totals[row["COUNTY"]]

        results.append({
            "Year": year,
            "Increment": row["INCREMENT"],
            "Statewide_Rank": row["STATEWIDE_RANK"],
            "Statewide_Percentile": row["STATEWIDE_PERCENTILE"],
            "County_Rank": row["COUNTY_RANK"],
            "County_Percentile": row["COUNTY_PERCENTILE"],
            "Age": row["AGE"],
            "Share_of_State": row["INCREMENT"] / statewide_total,
            "Share_of_County": row["INCREMENT"] / county_total
        })
    else:
        print(f"WARNING: Brookfield TID1 not uniquely found for {year}, found {len(tid1)} rows")

# Final dataframe
if results:
    brookfield_timeline = pd.DataFrame(results)
    brookfield_timeline = brookfield_timeline.sort_values("Year")
    print("\nBrookfield TID1 Timeline:")
    print(brookfield_timeline)

    # Export
    brookfield_timeline.to_excel("Brookfield_TID1_Timeline.xlsx", index=False)
    print("\nTimeline exported to Brookfield_TID1_Timeline.xlsx")
else:
    print("\nNo valid Brookfield TID1 data found in any year.")


Processing 2019...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2020...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2021...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2022...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YR.', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2023...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2024...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPALITY', 'TID_NUMBER', 'BASE_YEAR', 'CURRENT_VALUE', 'BASE_VALUE', 'INCREMENT']

Processing 2025...
Normalized columns: ['COUNTY', 'COMUN', 'TVC', 'MUNICIPAL

This code reads, cleans, and combines TIF data from 2019–2025 into a single dataset, adding a year column so all districts can be analyzed together and exported to Excel.

In [33]:
import pandas as pd

# Files to process
files = {
    2019: "2019_TIF_Raw_Data.xlsx",
    2020: "2020_TIF_Raw_Data.xlsx",
    2021: "2021_TIF_Raw_Data.xlsx",
    2022: "2022_TIF_Raw_Data.xlsx",
    2023: "2023_TIF_Raw_Data.xlsx",
    2024: "2024_TIF_Raw_Data.xlsx",
    2025: "2025_TIF_Raw_Data.xlsx"
}

all_data = []

for year, file in files.items():
    print(f"Processing {year}...")
    
    df = pd.read_excel(file)
    
    # Normalize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.replace("#", "NUMBER", regex=False)
        .str.replace(" ", "_", regex=False)
        .str.upper()
    )
    
    # Map any Base Year variant to BASE_YEAR
    base_year_candidates = [col for col in df.columns if "BASE" in col and ("YR" in col or "YEAR" in col)]
    if base_year_candidates:
        df = df.rename(columns={base_year_candidates[0]: "BASE_YEAR"})
    
    # Convert key identifier columns to string
    for col in ["COMUN", "TID_NUMBER", "COUNTY", "MUNICIPALITY", "TVC"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    
    # Convert numeric columns
    for col in ["BASE_YEAR", "INCREMENT", "BASE_VALUE", "CURRENT_VALUE"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # Do NOT drop rows — preserve all TIDs even if INCREMENT is missing
    # Fill INCREMENT missing values if desired (optional)
    # df["INCREMENT"] = df["INCREMENT"].fillna(0)
    
    # Add year column
    df["YEAR"] = year
    
    # Append to list
    all_data.append(df)

# Combine all years
tif_data_by_year = pd.concat(all_data, ignore_index=True)

# Optional: reorder columns for consistency
cols_order = ["COUNTY", "COMUN", "TVC", "MUNICIPALITY", "TID_NUMBER", "BASE_YEAR",
              "CURRENT_VALUE", "BASE_VALUE", "INCREMENT", "YEAR"]
tif_data_by_year = tif_data_by_year[[c for c in cols_order if c in tif_data_by_year.columns]]

# Export to Excel
tif_data_by_year.to_excel("TIF_Data_By_Year.xlsx", index=False)
print("\n✅ Exported combined data to TIF_Data_By_Year.xlsx with all rows preserved")

Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...
Processing 2025...

✅ Exported combined data to TIF_Data_By_Year.xlsx with all rows preserved


In [7]:
%pip install tableau-scraper
from tableauscraper import TableauScraper as TS
import pandas as pd
dashboard_url = "https://public.tableau.com/app/profile/research.policy/viz/TIDData0_1/TIDDataVisualization"

ts = TS()
ts.loads(dashboard_url)

worksheets = ts.getWorkbook().worksheets

print(f"Worksheets found: {len(worksheets)}")

for ws in worksheets:
    print(ws.name)

ERROR: Could not find a version that satisfies the requirement tableau-scraper (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Users/juanlasso/dataProjects/my-installs/venv/bin/python -m pip install --upgrade pip
ERROR: No matching distribution found for tableau-scraper
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'tableauscraper'

In [9]:
%pip install tableauscraper pandas --upgrade

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 13.7 MB/s  0:00:00 14.0 MB/s eta 0:00:01
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.0
    Uninstalling pandas-3.0.0:
      Successfully uninstalled pandas-3.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [tableauscraper] 2/5 [pandas]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Users/juanlasso/dataProjects/my-installs/venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
from tableauscraper import TableauScraper as TS
import pandas as pd

dashboard_url = "https://public.tableau.com/app/profile/research.policy/viz/TIDData0_1/TIDDataVisualization"

ts = TS()
ts.loads(dashboard_url)

workbook = ts.getWorkbook()
worksheets = workbook.worksheets

print(f"Worksheets found: {len(worksheets)}")

for ws in worksheets:
    print(ws.name)
    print(ws.data.head())  # Shows the first few rows of each worksheet

AttributeError: 'NoneType' object has no attribute 'text'

In [13]:
import pandas as pd

# -----------------------------------------------------
# 1. LOAD DATA (Tableau export encoding)
# -----------------------------------------------------

df = pd.read_csv(
    "Municipal_TID_Raw_Data_Tableau.csv",
    encoding="utf-16",
    sep="\t"
)

print("Raw dataset shape:", df.shape)

# -----------------------------------------------------
# 2. CLEAN COLUMN NAMES
# -----------------------------------------------------

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

print("\nColumns:")
print(df.columns)

# -----------------------------------------------------
# 3. FILTER FOR TOWN OF BROOKFIELD (67002)
# -----------------------------------------------------

brookfield = df[df["Municipality"] == "67002 BROOKFIELD (T)"]

print("\nTown of Brookfield rows:", brookfield.shape)

# -----------------------------------------------------
# 4. KEEP ONLY INCREMENT METRIC
# -----------------------------------------------------

increment_df = brookfield[brookfield["Metric"] == "Increment"]

print("Increment rows:", increment_df.shape)

# -----------------------------------------------------
# 5. SELECT IMPORTANT COLUMNS
# -----------------------------------------------------

increment_df = increment_df[
    [
        "Tax_Year",
        "TID_Name",
        "Current_Value",
        "Metric_Values",
        "TID_Type"
    ]
]

# -----------------------------------------------------
# 6. CONVERT NUMERIC FIELDS
# -----------------------------------------------------

increment_df["Tax_Year"] = pd.to_numeric(increment_df["Tax_Year"], errors="coerce")
increment_df["Current_Value"] = pd.to_numeric(increment_df["Current_Value"], errors="coerce")
increment_df["Metric_Values"] = pd.to_numeric(increment_df["Metric_Values"], errors="coerce")

# -----------------------------------------------------
# 7. SORT BY YEAR
# -----------------------------------------------------

increment_df = increment_df.sort_values(["TID_Name", "Tax_Year"])

print("\nCleaned dataset preview:")
print(increment_df.head())

# -----------------------------------------------------
# 8. CREATE PIVOT TABLE (TID per column)
# -----------------------------------------------------

tid_pivot = increment_df.pivot_table(
    index="Tax_Year",
    columns="TID_Name",
    values="Metric_Values",
    aggfunc="sum"
).reset_index()

tid_pivot = tid_pivot.sort_values("Tax_Year")

print("\nPivot dataset shape:", tid_pivot.shape)

# -----------------------------------------------------
# 9. SAVE DATASETS
# -----------------------------------------------------

increment_df.to_csv("brookfield_tid_long_format_2014_2025.csv", index=False)

tid_pivot.to_csv("brookfield_tid_increment_pivot_2014_2025.csv", index=False)

print("\nFiles saved:")
print("brookfield_tid_long_format_2014_2025.csv")
print("brookfield_tid_increment_pivot_2014_2025.csv")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte

In [15]:
import pandas as pd

# -----------------------------------------------------
# 1. LOAD DATA (Tableau export encoding)
# -----------------------------------------------------

df = pd.read_csv(
    "Municipal_TID_Raw_Data_Tableau.csv",
    encoding="utf-16",
    sep="\t"
)

print("Raw dataset shape:", df.shape)

# -----------------------------------------------------
# 2. CLEAN COLUMN NAMES
# -----------------------------------------------------

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

print("\nColumns:")
print(df.columns)

# -----------------------------------------------------
# 3. FILTER FOR TOWN OF BROOKFIELD (67002)
# -----------------------------------------------------

brookfield = df[df["Municipality"] == "67002 BROOKFIELD (T)"]

print("\nTown of Brookfield rows:", brookfield.shape)

# -----------------------------------------------------
# 4. KEEP ONLY INCREMENT METRIC
# -----------------------------------------------------

increment_df = brookfield[brookfield["Metric"] == "Increment"]

print("Increment rows:", increment_df.shape)

# -----------------------------------------------------
# 5. SELECT IMPORTANT COLUMNS
# -----------------------------------------------------

increment_df = increment_df[
    [
        "Tax_Year",
        "TID_Name",
        "Current_Value",
        "Metric_Values",
        "TID_Type"
    ]
]

# -----------------------------------------------------
# 6. CONVERT NUMERIC FIELDS
# -----------------------------------------------------

increment_df["Tax_Year"] = pd.to_numeric(increment_df["Tax_Year"], errors="coerce")
increment_df["Current_Value"] = pd.to_numeric(increment_df["Current_Value"], errors="coerce")
increment_df["Metric_Values"] = pd.to_numeric(increment_df["Metric_Values"], errors="coerce")

# -----------------------------------------------------
# 7. SORT BY YEAR
# -----------------------------------------------------

increment_df = increment_df.sort_values(["TID_Name", "Tax_Year"])

print("\nCleaned dataset preview:")
print(increment_df.head())

# -----------------------------------------------------
# 8. CREATE PIVOT TABLE (TID per column)
# -----------------------------------------------------

tid_pivot = increment_df.pivot_table(
    index="Tax_Year",
    columns="TID_Name",
    values="Metric_Values",
    aggfunc="sum"
).reset_index()

tid_pivot = tid_pivot.sort_values("Tax_Year")

print("\nPivot dataset shape:", tid_pivot.shape)

# -----------------------------------------------------
# 9. SAVE DATASETS
# -----------------------------------------------------

increment_df.to_csv("brookfield_tid_long_format_2014_2025.csv", index=False)

tid_pivot.to_csv("brookfield_tid_increment_pivot_2014_2025.csv", index=False)

print("\nFiles saved:")
print("brookfield_tid_long_format_2014_2025.csv")
print("brookfield_tid_increment_pivot_2014_2025.csv")

Raw dataset shape: (28360, 10)

Columns:
Index(['Tax_Year', 'Metric', 'TID_Name', 'Metric_(copy)', 'Municipality',
       'TID_Type', 'TID_Type_(2)', '#', 'Current_Value', 'Metric_Values'],
      dtype='str')

Town of Brookfield rows: (20, 10)
Increment rows: (0, 10)

Cleaned dataset preview:
Empty DataFrame
Columns: [Tax_Year, TID_Name, Current_Value, Metric_Values, TID_Type]
Index: []

Pivot dataset shape: (0, 1)

Files saved:
brookfield_tid_long_format_2014_2025.csv
brookfield_tid_increment_pivot_2014_2025.csv


In [17]:
import pandas as pd

# -----------------------------------------------------
# 1. LOAD DATA
# -----------------------------------------------------

df = pd.read_csv(
    "Municipal_TID_Raw_Data_Tableau.csv",
    encoding="utf-16",
    sep="\t"
)

print("Raw dataset shape:", df.shape)

# -----------------------------------------------------
# 2. CLEAN COLUMN NAMES
# -----------------------------------------------------

df.columns = df.columns.str.strip().str.replace(" ", "_")

print("\nColumns:")
print(df.columns)

# -----------------------------------------------------
# 3. KEEP IMPORTANT FIELDS
# -----------------------------------------------------

df = df[
    [
        "Tax_Year",
        "Metric",
        "TID_Name",
        "Municipality",
        "TID_Type",
        "Metric_Values"
    ]
]

# convert numbers
df["Metric_Values"] = pd.to_numeric(df["Metric_Values"], errors="coerce")
df["Tax_Year"] = pd.to_numeric(df["Tax_Year"], errors="coerce")

# -----------------------------------------------------
# 4. PIVOT METRICS INTO COLUMNS
# -----------------------------------------------------

tid_dataset = df.pivot_table(
    index=[
        "Tax_Year",
        "TID_Name",
        "Municipality",
        "TID_Type"
    ],
    columns="Metric",
    values="Metric_Values",
    aggfunc="first"
).reset_index()

# flatten column names
tid_dataset.columns.name = None

print("\nRebuilt dataset shape:", tid_dataset.shape)
print(tid_dataset.head())

# -----------------------------------------------------
# 5. SORT DATA
# -----------------------------------------------------

tid_dataset = tid_dataset.sort_values(["TID_Name", "Tax_Year"])

# -----------------------------------------------------
# 6. SAVE ORGANIZED DATASET
# -----------------------------------------------------

tid_dataset.to_csv("tid_dataset_organized_2014_2025.csv", index=False)

print("\nSaved: tid_dataset_organized_2014_2025.csv")


Raw dataset shape: (28360, 10)

Columns:
Index(['Tax_Year', 'Metric', 'TID_Name', 'Metric_(copy)', 'Municipality',
       'TID_Type', 'TID_Type_(2)', '#', 'Current_Value', 'Metric_Values'],
      dtype='str')

Rebuilt dataset shape: (13929, 6)
   Tax_Year        TID_Name          Municipality                    TID_Type  \
0      2014  ABBOTSFORD 005  10201 ABBOTSFORD (C)                   MIXED-USE   
1      2014       ADAMS 002       01201 ADAMS (C)  INDUSTRIAL AFTER 10/1/1995   
2      2014       ADAMS 003       01201 ADAMS (C)                      BLIGHT   
3      2014      ALBANY 002      23101 ALBANY (V)            BEFORE 10/1/1995   
4      2014      ALGOMA 001      31201 ALGOMA (C)        DISTRESSED MIXED-USE   

   Base Value  Increment   
0  11954100.0   4802500.0  
1   9585200.0   8102300.0  
2   5169700.0  10333800.0  
3   1209500.0   3527300.0  
4   7899200.0  -1192400.0  

Saved: tid_dataset_organized_2014_2025.csv


In [21]:
print(df.columns)

Index(['Tax_Year', 'TID_Name', 'Municipality', 'TID_Type', 'Base Value',
       'Increment ', 'CoMun', 'County'],
      dtype='str')


In [23]:
df.columns = df.columns.str.strip().str.replace(" ", "_")
print("Cleaned columns:", df.columns.tolist())

Cleaned columns: ['Tax_Year', 'TID_Name', 'Municipality', 'TID_Type', 'Base_Value', 'Increment', 'CoMun', 'County']


In [19]:
import pandas as pd

# -----------------------------------------------------
# 1. LOAD ORGANIZED DATASET
# -----------------------------------------------------

df = pd.read_csv("tid_dataset_organized_2014_2025.csv")

print("Dataset shape:", df.shape)

# -----------------------------------------------------
# 2. EXTRACT MUNICIPAL + COUNTY CODES
# -----------------------------------------------------

df["CoMun"] = df["Municipality"].str.split(" ").str[0]
df["County"] = df["CoMun"].str[:2]

# -----------------------------------------------------
# 3. FILTER YEARS (2014-2018)
# -----------------------------------------------------

df = df[(df["Tax_Year"] >= 2014) & (df["Tax_Year"] <= 2018)]

print("Filtered years shape:", df.shape)

# -----------------------------------------------------
# 4. STATEWIDE RANKINGS
# -----------------------------------------------------

df["Statewide_Rank"] = (
    df.groupby("Tax_Year")["Increment"]
    .rank(ascending=False, method="min")
)

df["Statewide_Percentile"] = (
    df.groupby("Tax_Year")["Increment"]
    .rank(pct=True)
)

# -----------------------------------------------------
# 5. COUNTY RANKINGS
# -----------------------------------------------------

df["County_Rank"] = (
    df.groupby(["Tax_Year","County"])["Increment"]
    .rank(ascending=False, method="min")
)

df["County_Percentile"] = (
    df.groupby(["Tax_Year","County"])["Increment"]
    .rank(pct=True)
)

# -----------------------------------------------------
# 6. FILTER TOWN OF BROOKFIELD
# -----------------------------------------------------

brookfield = df[df["Municipality"] == "67002 BROOKFIELD (T)"]

print("\nBrookfield rows:")
print(brookfield)

# -----------------------------------------------------
# 7. SAVE RESULTS
# -----------------------------------------------------

brookfield.to_csv(
    "brookfield_tid_rankings_2014_2018.csv",
    index=False
)

print("\nSaved: brookfield_tid_rankings_2014_2018.csv")

Dataset shape: (13929, 6)
Filtered years shape: (5890, 8)


KeyError: 'Column not found: Increment'

In [25]:
import pandas as pd

# -----------------------------------------------------
# 1. LOAD ORGANIZED DATASET
# -----------------------------------------------------

df = pd.read_csv("tid_dataset_organized_2014_2025.csv")

# Clean column names: strip spaces, replace spaces with underscores
df.columns = df.columns.str.strip().str.replace(" ", "_")

print("Dataset shape:", df.shape)
print("Cleaned columns:", df.columns.tolist())

# -----------------------------------------------------
# 2. EXTRACT MUNICIPAL + COUNTY CODES
# -----------------------------------------------------

# Ensure 'Municipality' exists after cleaning
if "Municipality" not in df.columns:
    raise KeyError("Column 'Municipality' not found after cleaning!")

df["CoMun"] = df["Municipality"].str.split(" ").str[0]
df["County"] = df["CoMun"].str[:2]

# -----------------------------------------------------
# 3. FILTER YEARS (2014-2018)
# -----------------------------------------------------

df = df[(df["Tax_Year"] >= 2014) & (df["Tax_Year"] <= 2018)]
print("Filtered years shape:", df.shape)

# -----------------------------------------------------
# 4. STATEWIDE RANKINGS
# -----------------------------------------------------

# Ensure 'Increment' exists
if "Increment" not in df.columns:
    raise KeyError("Column 'Increment' not found after cleaning!")

df["Statewide_Rank"] = (
    df.groupby("Tax_Year")["Increment"]
    .rank(ascending=False, method="min")
)

df["Statewide_Percentile"] = (
    df.groupby("Tax_Year")["Increment"]
    .rank(pct=True)
)

# -----------------------------------------------------
# 5. COUNTY RANKINGS
# -----------------------------------------------------

df["County_Rank"] = (
    df.groupby(["Tax_Year", "County"])["Increment"]
    .rank(ascending=False, method="min")
)

df["County_Percentile"] = (
    df.groupby(["Tax_Year", "County"])["Increment"]
    .rank(pct=True)
)

# -----------------------------------------------------
# 6. FILTER TOWN OF BROOKFIELD
# -----------------------------------------------------

brookfield = df[df["Municipality"] == "67002 BROOKFIELD (T)"]

print("\nBrookfield rows:")
print(brookfield.head())

# -----------------------------------------------------
# 7. SAVE RESULTS
# -----------------------------------------------------

brookfield.to_csv(
    "brookfield_tid_rankings_2014_2018.csv",
    index=False
)

print("\nSaved: brookfield_tid_rankings_2014_2018.csv")

Dataset shape: (13929, 6)
Cleaned columns: ['Tax_Year', 'TID_Name', 'Municipality', 'TID_Type', 'Base_Value', 'Increment']
Filtered years shape: (5890, 8)

Brookfield rows:
      Tax_Year         TID_Name          Municipality            TID_Type  \
1387      2015  BROOKFIELD 001A  67002 BROOKFIELD (T)  REHAB/CONSERVATION   
1388      2016  BROOKFIELD 001A  67002 BROOKFIELD (T)  REHAB/CONSERVATION   
1389      2017  BROOKFIELD 001A  67002 BROOKFIELD (T)  REHAB/CONSERVATION   
1390      2018  BROOKFIELD 001A  67002 BROOKFIELD (T)  REHAB/CONSERVATION   

      Base_Value    Increment  CoMun County  Statewide_Rank  \
1387  65986900.0   -1779100.0  67002     67          1104.0   
1388  65986900.0   30656600.0  67002     67           134.0   
1389  65986900.0  131697300.0  67002     67            20.0   
1390  65986900.0  257176200.0  67002     67             6.0   

      Statewide_Percentile  County_Rank  County_Percentile  
1387              0.022163         36.0           0.078947  
138

In [3]:
import pandas as pd

# -----------------------------------------------------
# 1. LOAD ORGANIZED DATASET
# -----------------------------------------------------

df = pd.read_csv("tid_dataset_organized_2014_2025.csv")

# Clean column names
df.columns = df.columns.str.strip().str.replace(" ", "_")

print("Dataset shape:", df.shape)
print("Columns:", df.columns)

# -----------------------------------------------------
# 2. EXTRACT MUNICIPAL + COUNTY CODES
# -----------------------------------------------------

df["CoMun"] = df["Municipality"].str.split(" ").str[0]
df["County"] = df["CoMun"].str[:2]

# -----------------------------------------------------
# 3. FILTER YEARS (2014-2018)
# -----------------------------------------------------

df = df[(df["Tax_Year"] >= 2014) & (df["Tax_Year"] <= 2018)]

print("Filtered years shape:", df.shape)

# -----------------------------------------------------
# 4. STATE TOTALS
# -----------------------------------------------------

df["State_Total_Increment"] = df.groupby("Tax_Year")["Increment"].transform("sum")

df["Share_of_State"] = (
    df["Increment"] / df["State_Total_Increment"]
)

# -----------------------------------------------------
# 5. COUNTY TOTALS
# -----------------------------------------------------

df["County_Total_Increment"] = (
    df.groupby(["Tax_Year", "County"])["Increment"]
    .transform("sum")
)

df["Share_of_County"] = (
    df["Increment"] / df["County_Total_Increment"]
)

# -----------------------------------------------------
# 6. STATEWIDE RANKINGS
# -----------------------------------------------------

df["Statewide_Rank"] = (
    df.groupby("Tax_Year")["Increment"]
    .rank(ascending=False, method="min")
)

df["Statewide_Percentile"] = (
    df.groupby("Tax_Year")["Increment"]
    .rank(pct=True)
)

# -----------------------------------------------------
# 7. COUNTY RANKINGS
# -----------------------------------------------------

df["County_Rank"] = (
    df.groupby(["Tax_Year","County"])["Increment"]
    .rank(ascending=False, method="min")
)

df["County_Percentile"] = (
    df.groupby(["Tax_Year","County"])["Increment"]
    .rank(pct=True)
)

# -----------------------------------------------------
# 8. FILTER TOWN OF BROOKFIELD
# -----------------------------------------------------

brookfield = df[df["Municipality"] == "67002 BROOKFIELD (T)"]

print("\nBrookfield rows:")
print(brookfield.head())

# -----------------------------------------------------
# 9. SAVE RESULTS
# -----------------------------------------------------

brookfield.to_csv(
    "brookfield_tid_rankings_2014_2018.csv",
    index=False
)

print("\nSaved: brookfield_tid_rankings_2014_2018.csv")

Dataset shape: (13929, 6)
Columns: Index(['Tax_Year', 'TID_Name', 'Municipality', 'TID_Type', 'Base_Value',
       'Increment'],
      dtype='str')
Filtered years shape: (5890, 8)

Brookfield rows:
      Tax_Year         TID_Name          Municipality            TID_Type  \
1387      2015  BROOKFIELD 001A  67002 BROOKFIELD (T)  REHAB/CONSERVATION   
1388      2016  BROOKFIELD 001A  67002 BROOKFIELD (T)  REHAB/CONSERVATION   
1389      2017  BROOKFIELD 001A  67002 BROOKFIELD (T)  REHAB/CONSERVATION   
1390      2018  BROOKFIELD 001A  67002 BROOKFIELD (T)  REHAB/CONSERVATION   

      Base_Value    Increment  CoMun County  State_Total_Increment  \
1387  65986900.0   -1779100.0  67002     67           1.608323e+10   
1388  65986900.0   30656600.0  67002     67           1.703324e+10   
1389  65986900.0  131697300.0  67002     67           1.969042e+10   
1390  65986900.0  257176200.0  67002     67           2.080815e+10   

      Share_of_State  County_Total_Increment  Share_of_County  St

In [5]:
top20_statewide = (
    df.sort_values(["Tax_Year", "Increment"], ascending=[True, False])
      .groupby("Tax_Year")
      .head(20)
)

top20_statewide 

,Tax_Year,TID_Name,Municipality,TID_Type,Base_Value,Increment,CoMun,County,State_Total_Increment,Share_of_State,County_Total_Increment,Share_of_County,Statewide_Rank,Statewide_Percentile,County_Rank,County_Percentile
7147,2014,MIDDLETON 003,13255 MIDDLETON (C),SPECIAL LEGISLATION,63401800.0,392478800.0,13255,13,1.487367e+10,0.026387,2.187362e+09,0.179430,1.0,1.000000,1.0,1.000000
9417,2014,PLEASANT PRAIRIE 002,30174 PLEASANT PRAIRIE (V),INDUSTRIAL AFTER 10/1/1995,83014900.0,385518900.0,30174,30,1.487367e+10,0.025920,8.293523e+08,0.464843,2.0,0.999107,1.0,1.000000
12283,2014,VERONA 007,13286 VERONA (C),INDUSTRIAL AFTER 10/1/1995,320400.0,383211000.0,13286,13,1.487367e+10,0.025764,2.187362e+09,0.175193,3.0,0.998214,2.0,0.987805
5659,2014,LAKE DELTON 003,56146 LAKE DELTON (V),MIXED-USE,43963700.0,222295700.0,56146,56,1.487367e+10,0.014946,3.970948e+08,0.559805,4.0,0.997321,1.0,1.000000
3720,2014,GLENDALE 008,40231 GLENDALE (C),BLIGHT,73733700.0,214933400.0,40231,40,1.487367e+10,0.014451,2.501657e+09,0.085916,5.0,0.996429,1.0,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7399,2018,MILWAUKEE 056,40251 MILWAUKEE (C),BLIGHT,8958600.0,151413600.0,40251,40,2.080815e+10,0.007277,3.823342e+09,0.039602,16.0,0.987893,6.0,0.954545
4464,2018,HOBART 001,05126 HOBART (V),MIXED-USE,20991900.0,149727800.0,05126,05,2.080815e+10,0.007196,1.164134e+09,0.128617,17.0,0.987086,1.0,1.000000
5192,2018,KENOSHA 016,30241 KENOSHA (C),INDUSTRIAL AFTER 10/1/2004,1571900.0,148209800.0,30241,30,2.080815e+10,0.007123,1.521385e+09,0.097418,18.0,0.986279,2.0,0.961538
879,2018,BELOIT 010,53206 BELOIT (C),INDUSTRIAL AFTER 10/1/1995,1763400.0,140891700.0,53206,53,2.080815e+10,0.006771,6.125094e+08,0.230024,19.0,0.985472,1.0,1.000000


In [7]:
import pandas as pd

# -----------------------------------------------------
# 1. LOAD DATASET
# -----------------------------------------------------

df = pd.read_csv("tid_dataset_organized_2014_2025.csv")

# Clean column names
df.columns = df.columns.str.strip().str.replace(" ", "_")

# -----------------------------------------------------
# 2. EXTRACT COUNTY + MUNICIPAL CODE
# -----------------------------------------------------

df["CoMun"] = df["Municipality"].str.split(" ").str[0]
df["County"] = df["CoMun"].str[:2]

# -----------------------------------------------------
# 3. FILTER YEARS (2014–2025)
# -----------------------------------------------------

df = df[(df["Tax_Year"] >= 2014) & (df["Tax_Year"] <= 2025)]

# -----------------------------------------------------
# 4. STATEWIDE RANKINGS
# -----------------------------------------------------

df["Statewide_Rank"] = (
    df.groupby("Tax_Year")["Increment"]
    .rank(ascending=False, method="min")
)

df["Statewide_Percentile"] = (
    df.groupby("Tax_Year")["Increment"]
    .rank(pct=True)
)

# -----------------------------------------------------
# 5. CREATE TOP 20 DATAFRAME
# -----------------------------------------------------

top20_statewide = (
    df.sort_values(["Tax_Year", "Increment"], ascending=[True, False])
      .groupby("Tax_Year")
      .head(20)
)

# -----------------------------------------------------
# 6. FLAG BROOKFIELD TIDs
# -----------------------------------------------------

top20_statewide["Brookfield_TID"] = (
    top20_statewide["Municipality"] == "67002 BROOKFIELD (T)"
)

# -----------------------------------------------------
# 7. SORT OUTPUT
# -----------------------------------------------------

top20_statewide = top20_statewide.sort_values(
    ["Tax_Year", "Statewide_Rank"]
)

print(top20_statewide.head(40))

# -----------------------------------------------------
# 8. EXPORT FILES
# -----------------------------------------------------

top20_statewide.to_csv(
    "top20_wisconsin_tids_2014_2025.csv",
    index=False
)

top20_statewide.to_excel(
    "top20_wisconsin_tids_2014_2025.xlsx",
    index=False
)

print("\nFiles exported:")
print("top20_wisconsin_tids_2014_2025.csv")
print("top20_wisconsin_tids_2014_2025.xlsx")

       Tax_Year              TID_Name                Municipality  \
7147       2014         MIDDLETON 003         13255 MIDDLETON (C)   
9417       2014  PLEASANT PRAIRIE 002  30174 PLEASANT PRAIRIE (V)   
12283      2014            VERONA 007            13286 VERONA (C)   
5659       2014       LAKE DELTON 003       56146 LAKE DELTON (V)   
3720       2014          GLENDALE 008          40231 GLENDALE (C)   
13022      2014         WAUWATOSA 002         40291 WAUWATOSA (C)   
13442      2014            WESTON 001            37192 WESTON (V)   
2199       2014            CUDAHY 001            40211 CUDAHY (C)   
11714      2014        STURTEVANT 003        51181 STURTEVANT (V)   
6045       2014           MADISON 032           13251 MADISON (C)   
7257       2014         MILWAUKEE 022         40251 MILWAUKEE (C)   
1513       2014        BURLINGTON 003        51206 BURLINGTON (C)   
6022       2014           MADISON 025           13251 MADISON (C)   
8902       2014           OSHKOSH 

In [1]:
import os
print(os.getcwd())

/Users/juanlasso/dataProjects/Milwaukee_Data/TID_Analysis 


In [5]:
import pandas as pd 
df = pd.read_csv("Municipal_TID_Values_2000_2025.csv", encoding="utf-16")

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28845 entries, 0 to 28844
Data columns (total 1 columns):
 #   Column                                                                             Non-Null Count  Dtype
---  ------                                                                             --------------  -----
 0   Tax Year	Measure Names	Metric	TID Name	Calculation1	#	Current Value	Metric Values  28845 non-null  str  
dtypes: str(1)
memory usage: 225.5 KB


In [9]:
import pandas as pd

df = pd.read_csv(
    "Municipal_TID_Values_2000_2025.csv",
    sep="\t",          # 👈 THIS is the key fix
    encoding="utf-16"  # keep this if it worked earlier
)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28845 entries, 0 to 28844
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Tax Year       28845 non-null  int64  
 1   Measure Names  52 non-null     str    
 2   Metric         52 non-null     str    
 3   TID Name       28793 non-null  str    
 4   Calculation1   50 non-null     str    
 5   #              27441 non-null  float64
 6   Current Value  52 non-null     float64
 7   Metric Values  28793 non-null  float64
dtypes: float64(3), int64(1), str(4)
memory usage: 1.8 MB


In [11]:
df["Measure Names"].value_counts()

Measure Names
Current Value    52
Name: count, dtype: int64

In [19]:
import pandas as pd

# Step 1: Load the data
df = pd.read_csv(
    "Municipal_TID_Values_2000_2025.csv",
    sep="\t",
    encoding="utf-16"
)

# Step 2: Pivot metrics into columns (exclude Current Value from index)
pivot_df = df.pivot_table(
    index=["Tax Year", "TID Name", "Municipality", "#"],
    columns="Metric",
    values="Metric Values",
    aggfunc="first"
).reset_index()

# Step 3: Ensure numeric
for col in ["Base Value", "Increment"]:
    if col in pivot_df.columns:
        pivot_df[col] = pd.to_numeric(pivot_df[col], errors="coerce")
    else:
        pivot_df[col] = 0

# Step 4: Determine TRUE Base Year for each TID
true_base_years = df.groupby("TID Name")["Tax Year"].min().reset_index()
true_base_years = true_base_years.rename(columns={"Tax Year": "Base Year"})

pivot_df = pivot_df.merge(true_base_years, on="TID Name", how="left")

# Step 5: Rename columns
pivot_df = pivot_df.rename(columns={
    "Municipality": "County Municipality",
    "#": "TID #"
})

# Step 6: Ensure Current Value exists
if "Current Value" not in pivot_df.columns:
    pivot_df["Current Value"] = pd.to_numeric(df["Current Value"], errors="coerce")

# Step 7: Compute Increment dynamically
pivot_df["Increment"] = pivot_df["Current Value"] - pivot_df["Base Value"]

# Step 8: Reorder columns
pivot_df = pivot_df[
    [
        "County Municipality",
        "TID #",
        "Base Year",
        "Tax Year",        # keep all years
        "Current Value",
        "Base Value",
        "Increment"
    ]
]

# Step 9: Export to Excel
pivot_df.to_excel("NEW_TID_Summary_By_Year_2000_2025.xlsx", index=False)

print("✅ Exported pivot table to 'TID_Summary_By_Year_2000_2025.xlsx'")

✅ Exported pivot table to 'TID_Summary_By_Year_2000_2025.xlsx'


This code takes a raw TID CSV and transforms it into a clean, year-by-year dataset. It pivots the metric values so that Base Value, Increment, and other metrics become separate columns, merges in the true Current Value per TID and Tax Year, ensures numeric calculations for Base Value and Increment, determines the first year each TID was active, renames and reorders columns to match the Master dataset, and finally exports the fully normalized dataset to Excel for analysis or comparison.

In [21]:
import pandas as pd

# Step 1: Load the data
df = pd.read_csv(
    "Municipal_TID_Values_2000_2025.csv",
    sep="\t",
    encoding="utf-16"
)

# Step 2: Pivot only the Metric Values (Base Value, Increment, etc.)
metrics_df = df.pivot_table(
    index=["Tax Year", "TID Name", "Municipality", "#"],
    columns="Metric",
    values="Metric Values",
    aggfunc="first"
).reset_index()

# Step 3: Merge the true Current Value per Tax Year and TID
current_values = df[["Tax Year", "TID Name", "Current Value"]].drop_duplicates()
current_values["Current Value"] = pd.to_numeric(current_values["Current Value"], errors="coerce")

full_df = metrics_df.merge(current_values, on=["Tax Year", "TID Name"], how="left")

# Step 4: Ensure numeric Base Value
full_df["Base Value"] = pd.to_numeric(full_df.get("Base Value", 0), errors="coerce")

# Step 5: Compute Increment dynamically
full_df["Increment"] = full_df["Current Value"] - full_df["Base Value"]

# Step 6: Determine first year the TID appears in dataset
start_years = df.groupby("TID Name")["Tax Year"].min().reset_index()
start_years = start_years.rename(columns={"Tax Year": "Start Year (TID Active)"})
full_df = full_df.merge(start_years, on="TID Name", how="left")

# Step 7: Rename columns for clarity
full_df = full_df.rename(columns={
    "Municipality": "County Municipality",
    "#": "TID #"
})

# Step 8: Reorder columns
full_df = full_df[
    [
        "County Municipality",
        "TID #",
        "Start Year (TID Active)",  # renamed from Base Year
        "Tax Year",
        "Current Value",
        "Base Value",
        "Increment"
    ]
]

# Step 9: Export to Excel
full_df.to_excel("TID_Full_By_Year.xlsx", index=False)

print("✅ Exported TID data to 'TID_Full_By_Year.xlsx' with Start Year (TID Active)")

✅ Exported TID data to 'TID_Full_By_Year.xlsx' with Start Year (TID Active)


In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58624 entries, 0 to 58623
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Tax Year       58624 non-null  int64  
 1   Metric         58624 non-null  str    
 2   TID Name       58624 non-null  str    
 3   Metric (copy)  58624 non-null  str    
 4   Municipality   58624 non-null  str    
 5   TID Type       58624 non-null  str    
 6   TID Type (2)   58624 non-null  str    
 7   #              55888 non-null  float64
 8   Current Value  58624 non-null  int64  
 9   Metric Values  58622 non-null  float64
dtypes: float64(2), int64(2), str(6)
memory usage: 4.5 MB


In [25]:
full_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28970 entries, 0 to 28969
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   County Municipality      28970 non-null  str    
 1   TID #                    28970 non-null  float64
 2   Start Year (TID Active)  28970 non-null  int64  
 3   Tax Year                 28970 non-null  int64  
 4   Current Value            28970 non-null  int64  
 5   Base Value               28967 non-null  float64
 6   Increment                28967 non-null  float64
dtypes: float64(3), int64(3), str(1)
memory usage: 1.5 MB


In [45]:
import pandas as pd

# Files to process
files = {
    2019: "2019_TIF_Raw_Data.xlsx",
    2020: "2020_TIF_Raw_Data.xlsx",
    2021: "2021_TIF_Raw_Data.xlsx",
    2022: "2022_TIF_Raw_Data.xlsx",
    2023: "2023_TIF_Raw_Data.xlsx",
    2024: "2024_TIF_Raw_Data.xlsx",
    2025: "2025_TIF_Raw_Data.xlsx"
}

all_data = []

for year, file in files.items():
    print(f"Processing {year}...")
    
    df = pd.read_excel(file)
    
    # Normalize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.replace("#", "NUMBER", regex=False)
        .str.replace(" ", "_", regex=False)
        .str.upper()
    )
    
    # Map any Base Year variant to BASE_YEAR
    base_year_candidates = [col for col in df.columns if "BASE" in col and ("YR" in col or "YEAR" in col)]
    if base_year_candidates:
        df = df.rename(columns={base_year_candidates[0]: "BASE_YEAR"})
    
    # Convert key identifier columns to string
    for col in ["COMUN", "TID_NUMBER", "COUNTY", "MUNICIPALITY", "TVC"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    
    # Convert numeric columns and remove commas
    for col in ["BASE_YEAR", "INCREMENT", "BASE_VALUE", "CURRENT_VALUE"]:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)                # Convert everything to string first
                .str.replace(",", "")       # Remove commas
                .replace("nan", pd.NA)     # Convert literal 'nan' strings to NA
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")  # Convert to numeric
    
    # Do NOT drop rows — preserve all TIDs even if INCREMENT is missing
    # Optional: fill missing increment with 0
    # df["INCREMENT"] = df["INCREMENT"].fillna(0)
    
    # Add year column
    df["YEAR"] = year
    
    # Append to list
    all_data.append(df)

# Combine all years
tif_data_by_year = pd.concat(all_data, ignore_index=True)

# Optional: reorder columns for consistency
cols_order = ["COUNTY", "COMUN", "TVC", "MUNICIPALITY", "TID_NUMBER", "BASE_YEAR",
              "CURRENT_VALUE", "BASE_VALUE", "INCREMENT", "YEAR"]
tif_data_by_year = tif_data_by_year[[c for c in cols_order if c in tif_data_by_year.columns]]

# Export to Excel
tif_data_by_year.to_excel("TIF_Data_By_Year.xlsx", index=False)
print("\n✅ Exported combined data to TIF_Data_By_Year.xlsx with all rows preserved")

Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...
Processing 2025...

✅ Exported combined data to TIF_Data_By_Year.xlsx with all rows preserved


In [64]:
import pandas as pd

# Load datasets
master_df = pd.read_excel("Master_TID_By_Year_2000_2025.xlsx")
full_df = pd.read_excel("TIF_Data_By_Year.xlsx")

# Filter for years 2019–2025 (optional, can adjust for a single year)
master_filtered = master_df[(master_df["Tax Year"] >= 2019) & (master_df["Tax Year"] <= 2025)].copy()
full_filtered = full_df[(full_df["YEAR"] >= 2019) & (full_df["YEAR"] <= 2025)].copy()

# Normalize Current Value: remove commas, convert to numeric
master_filtered["Current_Value_Num"] = master_filtered["Current Value"].astype(str).str.replace(",", "").astype(float)
full_filtered["Current_Value_Num"] = full_filtered["CURRENT_VALUE"].astype(str).str.replace(",", "").astype(float)

# Create a key using Tax Year + Current Value
master_filtered["KEY"] = master_filtered["Tax Year"].astype(str) + "_" + master_filtered["Current_Value_Num"].astype(str)
full_filtered["KEY"] = full_filtered["YEAR"].astype(str) + "_" + full_filtered["Current_Value_Num"].astype(str)

# Identify missing keys (Master rows not in Full)
missing_keys = set(master_filtered["KEY"]) - set(full_filtered["KEY"])
missing_rows = master_filtered[master_filtered["KEY"].isin(missing_keys)].copy()

# Identify extra rows in Full (just to check)
extra_keys = set(full_filtered["KEY"]) - set(master_filtered["KEY"])
extra_rows = full_filtered[full_filtered["KEY"].isin(extra_keys)].copy()

# Summary
print(f"Master rows (2019-2025): {len(master_filtered)}")
print(f"Full By Year rows (2019-2025): {len(full_filtered)}")
print(f"Rows missing in Full By Year: {len(missing_rows)}")
print(f"Rows extra in Full By Year: {len(extra_rows)}")

# Optional: export for inspection
missing_rows.to_excel("Missing_Rows_TaxYear_CurrentValue.xlsx", index=False)
extra_rows.to_excel("Extra_Rows_TaxYear_CurrentValue.xlsx", index=False)
print("\n✅ Exported missing and extra rows for inspection")

Master rows (2019-2025): 9951
Full By Year rows (2019-2025): 9621
Rows missing in Full By Year: 7
Rows extra in Full By Year: 9

✅ Exported missing and extra rows for inspection


In [66]:
missing_summary = missing_rows.groupby("Tax Year").size().reset_index(name="Missing Count")
print(missing_summary)

   Tax Year  Missing Count
0      2020              2
1      2021              4
2      2023              1


In [68]:
import pandas as pd

# Step 1: Load the datasets
master_df = pd.read_excel("Master_TID_By_Year_2000_2025.xlsx")
full_df = pd.read_excel("TIF_Data_By_Year_2019_2025.xlsx")

# Step 2: Filter for 2025
master_2025 = master_df[master_df["Tax Year"] == 2025].copy()
full_2025 = full_df[full_df["YEAR"] == 2025].copy()

# Step 3: Normalize Current Value (remove commas, convert to numeric)
master_2025["Current_Value_Num"] = master_2025["Current Value"].astype(str).str.replace(",", "").astype(float)
full_2025["Current_Value_Num"] = full_2025["CURRENT_VALUE"].astype(str).str.replace(",", "").astype(float)

# Step 4: Create matching key (Tax Year + Current Value)
master_2025["KEY"] = master_2025["Tax Year"].astype(str) + "_" + master_2025["Current_Value_Num"].astype(str)
full_2025["KEY"] = full_2025["YEAR"].astype(str) + "_" + full_2025["Current_Value_Num"].astype(str)

# Step 5: Merge to flag matches
merged_2025 = master_2025.merge(
    full_2025[["KEY"]],
    on="KEY",
    how="left",
    indicator="Match_Flag"
)

# Step 6: Convert merge indicator to Yes/No
merged_2025["Found_in_Both"] = merged_2025["Match_Flag"].apply(lambda x: "Yes" if x == "both" else "No")

# Step 7: Optional: keep only relevant columns
columns_to_keep = [
    "County Municipality",
    "TID #",
    "Start Year (TID Active)",
    "Tax Year",
    "Current Value",
    "Base Value",
    "Increment",
    "Found_in_Both"
]
merged_2025 = merged_2025[columns_to_keep]

# Step 8: Export to Excel
merged_2025.to_excel("TID_2025_Reconciliation.xlsx", index=False)

print("✅ Reconciliation for 2025 complete. 'Found_in_Both' column added. Exported to 'TID_2025_Reconciliation.xlsx'")

✅ Reconciliation for 2025 complete. 'Found_in_Both' column added. Exported to 'TID_2025_Reconciliation.xlsx'


In [70]:
import pandas as pd

# Step 1: Load datasets
master_df = pd.read_excel("Master_TID_By_Year_2000_2025.xlsx")
full_df = pd.read_excel("TIF_Data_By_Year_2019_2025.xlsx")

# Step 2: Filter for 2025
master_2025 = master_df[master_df["Tax Year"] == 2025].copy()
full_2025 = full_df[full_df["YEAR"] == 2025].copy()

# Step 3: Normalize Current Value
master_2025["Current_Value_Num"] = master_2025["Current Value"].astype(str).str.replace(",", "").astype(float)
full_2025["Current_Value_Num"] = full_2025["CURRENT_VALUE"].astype(str).str.replace(",", "").astype(float)

# Step 4: Create key using Tax Year + Current Value
master_2025["KEY"] = master_2025["Tax Year"].astype(str) + "_" + master_2025["Current_Value_Num"].astype(str)
full_2025["KEY"] = full_2025["YEAR"].astype(str) + "_" + full_2025["Current_Value_Num"].astype(str)

# Step 5: Full outer merge on KEY
merged_2025 = pd.merge(
    master_2025,
    full_2025[["KEY"]],
    on="KEY",
    how="outer",
    indicator=True
)

# Step 6: Create a column for Yes/No/Only in Master/Only in Full
def flag_merge(row):
    if row["_merge"] == "both":
        return "Yes (in both)"
    elif row["_merge"] == "left_only":
        return "Only in Master"
    else:
        return "Only in Full"

merged_2025["Found_in_Both"] = merged_2025.apply(flag_merge, axis=1)

# Step 7: Keep relevant columns
columns_to_keep = [
    "County Municipality",
    "TID #",
    "Start Year (TID Active)",
    "Tax Year",
    "Current Value",
    "Base Value",
    "Increment",
    "Found_in_Both"
]
# Some columns may be NaN for outer merge, fill with placeholders if needed
for col in columns_to_keep:
    if col not in merged_2025.columns:
        merged_2025[col] = None

merged_2025 = merged_2025[columns_to_keep]

# Step 8: Export
merged_2025.to_excel("TID_2025_Full_Outer_Reconciliation.xlsx", index=False)

print("✅ Full outer merge for 2025 complete. 'Found_in_Both' column added. Exported to 'TID_2025_Full_Outer_Reconciliation.xlsx'")

✅ Full outer merge for 2025 complete. 'Found_in_Both' column added. Exported to 'TID_2025_Full_Outer_Reconciliation.xlsx'
